<a href="https://colab.research.google.com/github/Shacxify/prompt-engineering-exercises/blob/main/01_prompt_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 - Prompt Chaining for a Customer Support AI

**Cash Johnson | BUS4 118S - Agentic AI for Business | Prof. Haubrich**

**Tools used:** Google Colab + Google Gemini API (`google-genai` SDK, model `gemini-2.5-flash`).
No orchestration framework. LangChain and LangBase both do prompt chaining for you, and that is
the problem for this exercise: the dependency between steps is the thing being graded, so hiding it
inside `SequentialChain` would hide the assignment. Every handoff here is a visible Python variable.

**Goal:** a four-step prompt chain that runs a support ticket end to end, where each step's output is
the next step's input. Nothing is pasted by hand between steps.

**Scenario:** VNTG OS, the two-sided vintage consignment app I built for BUS4 110B. Two sides write
in: consignors who dropped off clothing, and buyers who bought it.

---

### Chain map

| Step | Name | Input | Output | Consumed by |
|---|---|---|---|---|
| 1 | Classify | raw ticket text | JSON: `category`, `side`, `urgency`, `missing_info[]`, `payout_at_risk_usd` | steps 2, 3, 4 |
| 2 | Gather | step 1 `missing_info[]` | the exact clarifying questions, one per missing field | step 3 |
| 3 | Resolve | step 1 JSON + answers + **policy retrieved by step 1's category** | resolution + reply draft | step 4 |
| 4 | Route | step 1 + step 3 + ticket | deterministic escalation rule, human handoff note if it fires | agent inbox |

### Techniques from the module used in this notebook

| Module technique | Where it shows up here |
|---|---|
| **Prompt chaining** | the whole notebook: 4 steps, each consuming the last |
| **Context-Aware Decomposition (CAD)** | one support ticket broken into classify / gather / resolve / route, each step aware of the overall goal |
| **System prompt vs user prompt** | every step has both, labeled, with the role and constraints in the system prompt |
| **Role-based prompting** | "intake classifier", "senior support specialist", "internal handoff writer" |
| **Clarity and specificity** | literal JSON schema and enums in step 1 instead of "tell me what the issue is" |
| **Content structuring** | every step specifies its output format, section by section |
| **Negative prompting** | "do not invent details", "do not cite a policy rule not in the block", "no greeting, no sign-off" |
| **Few-shot prompting with instruction** | step 2, tested head to head against the zero-shot version |
| **A/B testing** | section 5: two versions of step 2, same input, measured on three checks, winner used downstream |
| **Including contextual data (RAG-style)** | step 3 retrieves only the policy lines matching step 1's category, not the whole policy |
| **Temperature** | 0 on every classification step so the same ticket routes the same way twice |
| **Learning from failed prompts** | section 9: four failures and what each one changed |

In [8]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
import json

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-2.5-flash'

# Confirm the model name is live on this key before the chain runs.
available = [m.name for m in client.models.list()]
assert any(MODEL in m for m in available), f'{MODEL} not available. Options: {available[:10]}'
print('Connected. Using:', MODEL)

def ask(user_prompt, system=None, temperature=0.0, json_mode=False, model=MODEL):
    """One call to Gemini.

    system      -> the SYSTEM prompt: role, constraints, rules of engagement
    user_prompt -> the USER prompt: the task and the data for this specific call
    temperature -> 0 by default, so the chain routes the same ticket the same way every run
    """
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    if json_mode:
        kwargs['response_mime_type'] = 'application/json'
    resp = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(**kwargs),
    )
    return (resp.text or '').strip()

print('helper ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
Connected. Using: gemini-2.5-flash
helper ready


In [9]:
# =====================================================================
# 1. THE INPUTS, and the policy base the chain retrieves from
# =====================================================================
# Ticket A is the walkthrough. Ticket B runs through the same chain at
# the end to prove the escalation rule actually fires.

TICKET_A = """Subject: still waiting??

hey so i dropped a bunch of stuff off at the store like 3 weeks ago and nothing has
shown up on my account yet. one of them was a carhartt detroit jacket that should be
worth a lot. also my last payout was way less than i expected. can someone look at this

- marcus"""

TICKET_B = """Subject: CHARGED TWICE

I ordered the Ralph Lauren rugby ($480) on Tuesday and my bank shows two charges for the
same amount 4 minutes apart. I have already called my bank about disputing it. I need this
fixed today or I am filing the chargeback.

Danielle R."""

print(TICKET_A)

# ---------------------------------------------------------------------
# TECHNIQUE: including contextual data (RAG-style grounding).
# The model is never asked what the policy is, because it does not know
# and will invent one. Policy is passed in as context, and the retrieval
# is keyed off the category step 1 produces, so step 3 sees only what
# applies to this ticket plus the rules that always apply.
# ---------------------------------------------------------------------

POLICY_INDEX = {
    'intake_delay': [
        'Dropped-off items are photographed, priced, and listed within 10 business days.',
        'Drop-off volume spikes may extend intake; the SLA does not pause, it is missed.',
        'A consignor may ask for the current status of any item by drop-off date.',
    ],
    'payout_dispute': [
        'Split: consignor 60% / store 40% of final sale price.',
        'Items sold at $200 or more pay the consignor 65%.',
        'Payouts run the 1st and the 15th, only on items past the 7-day buyer return window.',
        'A return reverses that item\u2019s payout line on the next cycle.',
        'Never promise a specific payout amount before the return window closes.',
    ],
    'billing_error': [
        'Duplicate charges are verified against the payment processor before any refund.',
        'Confirmed duplicates are refunded to the original method within 5 business days.',
        'Support never confirms a refund before the processor record is checked.',
    ],
    'item_condition': [
        'Buyers may return within 7 days for condition not matching the listing.',
        'Measurements are entered at intake and are not independently verified.',
        'A consignor may request one re-list at a new price per item.',
    ],
    'shipping': [
        'Local pickup and standard shipping only; no expedited service.',
        'Tracking is emailed at the time the label is created.',
    ],
    'account_access': [
        'Account recovery requires the email on file; support cannot change it over chat.',
    ],
    'other': [],
}

ALWAYS_APPLIES = [
    'Escalate to a human on any of: a payout dispute over $150, a suspected duplicate charge, '
    'any mention of a chargeback or bank dispute, or urgency = high.',
    'Do not promise timelines that are not written in policy.',
]

def retrieve_policy(category: str) -> str:
    """Return only the policy lines relevant to this ticket's category, plus the always-on rules."""
    lines = POLICY_INDEX.get(category, []) + ALWAYS_APPLIES
    return '\n'.join(f'- {l}' for l in lines)

TOTAL_LINES = sum(len(v) for v in POLICY_INDEX.values()) + len(ALWAYS_APPLIES)
print(f'{TOTAL_LINES} policy lines in the base\n')
print('retrieve_policy("payout_dispute") returns:\n')
print(retrieve_policy('payout_dispute'))

Subject: still waiting??

hey so i dropped a bunch of stuff off at the store like 3 weeks ago and nothing has
shown up on my account yet. one of them was a carhartt detroit jacket that should be
worth a lot. also my last payout was way less than i expected. can someone look at this

- marcus
19 policy lines in the base

retrieve_policy("payout_dispute") returns:

- Split: consignor 60% / store 40% of final sale price.
- Items sold at $200 or more pay the consignor 65%.
- Payouts run the 1st and the 15th, only on items past the 7-day buyer return window.
- A return reverses that item’s payout line on the next cycle.
- Never promise a specific payout amount before the return window closes.
- Escalate to a human on any of: a payout dispute over $150, a suspected duplicate charge, any mention of a chargeback or bank dispute, or urgency = high.
- Do not promise timelines that are not written in policy.


## 3. Step 1 v1 - the version that broke the chain

**Technique: learning from failed prompts.**

First attempt. It reads like a reasonable instruction, and the answer it returns is not wrong, it is
just unparseable. Step 2 needs to loop over a list of missing fields, and prose is not a list.

## 4. Step 1 v2 - classify

Same job, prompt rewritten. Three things changed and only three mattered.

I told it exactly which keys I wanted back, because step 2 indexes `missing_info` and a
renamed key breaks the chain. I gave `category` a fixed list to choose from, because that
string is the lookup key for the policy retrieval in step 3. And I set
`response_mime_type='application/json'`, which forces valid JSON at the API level instead
of asking nicely for it.

Everything else I left alone. Temperature stays at 0 so the same ticket routes the same way
twice, which matters when the thing downstream is a money decision.

In [10]:
## 3. Step 1 v1 - the version that broke the chain

**Technique: learning from failed prompts.**

First attempt. It reads like a reasonable instruction, and the answer it returns is not wrong, it is
just unparseable. Step 2 needs to loop over a list of missing fields, and prose is not a list.

## 4. Step 1 v2 - classify

Same job, prompt rewritten. Three things changed and only three mattered.

I told it exactly which keys I wanted back, because step 2 indexes `missing_info` and a
renamed key breaks the chain. I gave `category` a fixed list to choose from, because that
string is the lookup key for the policy retrieval in step 3. And I set
`response_mime_type='application/json'`, which forces valid JSON at the API level instead
of asking nicely for it.

Everything else I left alone. Temperature stays at 0 so the same ticket routes the same way
twice, which matters when the thing downstream is a money decision.

SyntaxError: invalid syntax (3832493487.py, line 3)